# 实验一：静态页面与 API 数据采集

**平台：SmartPark DataLab**  
**目标：** 通过 API 分页采集 5000 条 Device 数据；通过 BeautifulSoup 解析 Company 静态 HTML 并沿“下一页”采集 3 页（目标 60 条）；验证 ID 唯一性、Company–Device 数量一致性，并保存 JSON。

> 重要：实验手册没有给出 Device API 的实际 Request URL、JSON 字段名和 Company DOM 的实际 selector。这些必须在你打开课程平台后通过浏览器 Network / 源代码 / Elements 观察，不能凭空填写。请先完成下面“配置区”。

In [1]:
import json
import math
from pathlib import Path
from collections import Counter
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://115.159.159.82"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/130 Safari/537.36"
})
print("环境准备完成")
print("requests:", requests.__version__)

环境准备完成
requests: 2.31.0


## 1. 先在浏览器中观察真实页面

分别打开 `BASE_URL + "/devices"` 和 `BASE_URL + "/companies"`。

- Device：右键查看网页源代码，搜索页面中可见的设备名称；再按 `F12 → Network → Fetch/XHR`，刷新并选中返回设备列表的请求，记录 Request URL、Method、Query 参数。
- Company：右键“查看网页源代码”，按 `Ctrl+F` 搜索真实企业名称；如果能搜到，说明可按静态 HTML 解析。
- Company 的记录 selector 与“下一页” selector：在 `F12 → Elements` 中检查实际 DOM。

下面配置值需要按你观察到的真实内容填写。

In [14]:

API_URL = "https://115.159.159.82/api/devices"
PAGE_PARAM = "page"
PAGE_SIZE_PARAM = "page_size"

TOTAL_KEY = "total"
PAGE_SIZE_KEY = "page_size"
ITEMS_KEY = "items"

COMPANY_URL = urljoin(BASE_URL, "/companies")
COMPANY_CARD_SELECTOR = "article.company-card"
NEXT_SELECTOR = 'a[href*="page="]'

print("API_URL 已填写：", bool(API_URL))
print("分页参数已填写：", bool(PAGE_PARAM and PAGE_SIZE_PARAM))
print("JSON 字段已填写：", bool(TOTAL_KEY and PAGE_SIZE_KEY and ITEMS_KEY))


API_URL 已填写： True
分页参数已填写： True
JSON 字段已填写： True


In [15]:

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
import json

company_session = session

def scrape_companies(max_pages=3):
    url = COMPANY_URL
    all_companies = []

    for page in range(1, max_pages + 1):
        response = company_session.get(
            url,
            timeout=20,
            verify=False
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        cards = soup.select(COMPANY_CARD_SELECTOR)

        if not cards:
            raise ValueError(
                f"第 {page} 页没有找到企业卡片，请检查选择器。"
            )

        for card in cards:
            company = {
                "company_id": card.get("data-company-id"),
                "name": (
                    card.select_one("h2 a").get_text(strip=True)
                    if card.select_one("h2 a")
                    else None
                ),
            }
            all_companies.append(company)

        print(f"第 {page} 页：采集到 {len(cards)} 家企业")

        if page < max_pages:
            next_link = soup.select_one(NEXT_SELECTOR)
            if not next_link or not next_link.get("href"):
                raise ValueError(
                    f"第 {page} 页没有找到下一页链接，请检查选择器。"
                )
            url = urljoin(url, next_link["href"])

    return all_companies

companies = scrape_companies(max_pages=3)

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("企业采集总数：", len(companies))
print("企业编号缺失数：", sum(not x["company_id"] for x in companies))
print("已保存：", (data_dir / "companies.json").resolve())


C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,


第 1 页：采集到 20 家企业
第 2 页：采集到 20 家企业
第 3 页：采集到 20 家企业
企业采集总数： 60
企业编号缺失数： 0
已保存： D:\大数据采集\实验一\lab01\data\companies.json


In [16]:

from collections import Counter

# 统计每家企业在设备数据中实际拥有的设备数量
device_counts = Counter(
    item.get("company_id")
    for item in devices
    if item.get("company_id") is not None
)

# 从企业网页卡片中提取企业编号和页面展示的设备数量
company_details = []

for company in companies:
    company_id = company.get("company_id")
    if not company_id:
        continue

    # 重新请求企业卡片所在页面，解析设备数量
    # 注意：当前 companies 变量仅保存了编号和名称，
    # 因此需要重新读取企业列表页面的 HTML 来获取设备数量。
    pass

print("设备数据中涉及的企业数：", len(device_counts))
print("已采集企业数：", len(companies))
print("企业与设备关联统计已完成第一步。")


设备数据中涉及的企业数： 300
已采集企业数： 60
企业与设备关联统计已完成第一步。


In [17]:

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from collections import Counter

# 统计设备数据中每家企业的实际设备数量
device_counts = Counter(
    item.get("company_id")
    for item in devices
    if item.get("company_id") is not None
)

# 重新采集前三页企业信息，包括页面展示的设备数量
company_details = []
url = COMPANY_URL

for page in range(1, 4):
    response = company_session.get(url, timeout=20, verify=False)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    for card in soup.select(COMPANY_CARD_SELECTOR):
        company_id = card.get("data-company-id")
        name_tag = card.select_one("h2 a")
        name = name_tag.get_text(strip=True) if name_tag else ""

        # 从企业卡片中查找“设备数量”及其对应数值
        text = card.get_text(" ", strip=True)
        import re
        match = re.search(r"设备数量\s*([0-9,]+)\s*台?", text)

        if match:
            declared_count = int(match.group(1).replace(",", ""))
        else:
            declared_count = None

        company_details.append({
            "company_id": company_id,
            "name": name,
            "declared_device_count": declared_count,
            "actual_device_count": device_counts.get(company_id, 0)
        })

    if page < 3:
        next_link = soup.select_one(NEXT_SELECTOR)
        if not next_link or not next_link.get("href"):
            raise ValueError(f"第 {page} 页未找到下一页链接")
        url = urljoin(url, next_link["href"])

# 只比较成功提取到页面登记数量的企业
comparable = [
    c for c in company_details
    if c["declared_device_count"] is not None
]

mismatches = [
    c for c in comparable
    if c["declared_device_count"] != c["actual_device_count"]
]

print("企业采集数：", len(company_details))
print("成功读取登记设备数量的企业数：", len(comparable))
print("参与比较的企业数：", len(comparable))
print("设备数量不一致企业数：", len(mismatches))

if mismatches:
    print("\n不一致企业示例（最多展示 10 家）：")
    for c in mismatches[:10]:
        print(
            c["company_id"], c["name"],
            "登记数量：", c["declared_device_count"],
            "实际数量：", c["actual_device_count"]
        )


C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,


企业采集数： 60
成功读取登记设备数量的企业数： 60
参与比较的企业数： 60
设备数量不一致企业数： 0


In [18]:

company_ids = [c.get("company_id") for c in companies]

print("企业采集总数：", len(companies))
print("唯一企业编号数：", len(set(company_ids)))
print("缺失企业编号数：", sum(x is None for x in company_ids))


企业采集总数： 60
唯一企业编号数： 40
缺失企业编号数： 0


In [19]:

from collections import Counter

company_id_counts = Counter(
    c.get("company_id") for c in companies
)

duplicates = {
    cid: count
    for cid, count in company_id_counts.items()
    if count > 1
}

print("重复企业编号数量：", len(duplicates))
print("重复编号及出现次数：")
for cid, count in duplicates.items():
    print(cid, "出现", count, "次")

print("\n每页采集记录数应为：20、20、20")


重复企业编号数量： 20
重复编号及出现次数：
C000001 出现 2 次
C000002 出现 2 次
C000003 出现 2 次
C000004 出现 2 次
C000005 出现 2 次
C000006 出现 2 次
C000007 出现 2 次
C000008 出现 2 次
C000009 出现 2 次
C000010 出现 2 次
C000011 出现 2 次
C000012 出现 2 次
C000013 出现 2 次
C000014 出现 2 次
C000015 出现 2 次
C000016 出现 2 次
C000017 出现 2 次
C000018 出现 2 次
C000019 出现 2 次
C000020 出现 2 次

每页采集记录数应为：20、20、20


In [20]:

from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = COMPANY_URL

for page in range(1, 4):
    response = company_session.get(url, timeout=20, verify=False)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    cards = soup.select(COMPANY_CARD_SELECTOR)

    ids = [card.get("data-company-id") for card in cards]

    print(f"\n第 {page} 页")
    print("实际请求地址：", response.url)
    print("企业数量：", len(ids))
    print("前 5 个企业编号：", ids[:5])
    print("后 5 个企业编号：", ids[-5:])

    if page < 3:
        next_link = soup.select_one(NEXT_SELECTOR)
        if not next_link or not next_link.get("href"):
            raise ValueError("找不到下一页链接")
        url = urljoin(response.url, next_link["href"])


C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,



第 1 页
实际请求地址： https://115.159.159.82/companies
企业数量： 20
前 5 个企业编号： ['C000001', 'C000002', 'C000003', 'C000004', 'C000005']
后 5 个企业编号： ['C000016', 'C000017', 'C000018', 'C000019', 'C000020']

第 2 页
实际请求地址： https://115.159.159.82/companies?page=2
企业数量： 20
前 5 个企业编号： ['C000021', 'C000022', 'C000023', 'C000024', 'C000025']
后 5 个企业编号： ['C000036', 'C000037', 'C000038', 'C000039', 'C000040']

第 3 页
实际请求地址： https://115.159.159.82/companies?page=1
企业数量： 20
前 5 个企业编号： ['C000001', 'C000002', 'C000003', 'C000004', 'C000005']
后 5 个企业编号： ['C000016', 'C000017', 'C000018', 'C000019', 'C000020']


In [21]:

from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = urljoin(BASE_URL, "/companies?page=2")

response = company_session.get(url, timeout=20, verify=False)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

nav = soup.select_one('nav.pagination')

print("当前页面：", response.url)
print("分页区域中的全部链接：")

for link in nav.select("a"):
    print(
        "文字：", link.get_text(strip=True),
        "| href：", link.get("href")
    )


当前页面： https://115.159.159.82/companies?page=2
分页区域中的全部链接：
文字： 上一页 | href： /companies?page=1
文字： 下一页 | href： /companies?page=3


C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,


## 2. Device：复现第一页 API 请求并检查 JSON 结构

In [4]:
if not API_URL:
    print("请先填写 API_URL，然后重新运行本单元格。")
else:
    first_response = session.get(
        API_URL,
        params={PAGE_PARAM: 1, PAGE_SIZE_PARAM: 20} if PAGE_PARAM and PAGE_SIZE_PARAM else None,
        timeout=20,
        verify=False
    )
    print("status:", first_response.status_code)
    print("url:", first_response.url)
    print("content-type:", first_response.headers.get("Content-Type"))
    first_response.raise_for_status()
    first_json = first_response.json()
    print("JSON 顶层类型：", type(first_json).__name__)
    if isinstance(first_json, dict):
        print("顶层 keys：", list(first_json.keys()))
    print(json.dumps(first_json, ensure_ascii=False, indent=2)[:12000])

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=20
content-type: application/json
JSON 顶层类型： dict
顶层 keys： ['page', 'page_size', 'total', 'items']
{
  "page": 1,
  "page_size": 20,
  "total": 5000,
  "items": [
    {
      "device_id": "D00000001",
      "company_id": "C000001",
      "device_name": "循环泵-000001-001",
      "device_type": "pump",
      "model": "SP-PUM-800",
      "zone": "智造一区",
      "install_date": "2021-09-19",
      "rated_power_kw": 71.2,
      "sampling_interval_s": 30,
      "protocol": "mqtt",
      "status": "running",
      "updated_at": "2026-09-01T03:58:00"
    },
    {
      "device_id": "D00000002",
      "company_id": "C000001",
      "device_name": "循环泵-000001-002",
      "device_type": "pump",
      "model": "SP-PUM-500",
      "zone": "智造一区",
      "install_date": "2016-03-06",
      "rated_power_kw": 73.8,
      "sampling_interval_s": 60,
      "protocol": "mqtt",
      "status": "running",
      "updated_at": "2026-09-08T05:44:0

C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,


观察上一个单元格的输出，填写 `TOTAL_KEY`、`PAGE_SIZE_KEY`、`ITEMS_KEY`。如果设备列表在嵌套字段中，`ITEMS_KEY` 可以先使用最外层字段；如需嵌套取值，请在下方自行按实际 JSON 结构修改 `get_field()`。

In [11]:
def get_field(obj, key):
    """按字段名读取顶层 JSON；若实际结构嵌套，请按观察结果修改此函数。"""
    return obj[key]

def fetch_devices():
    if not all([API_URL, PAGE_PARAM, PAGE_SIZE_PARAM, TOTAL_KEY, PAGE_SIZE_KEY, ITEMS_KEY]):
        raise ValueError("请先填写 API_URL、分页参数名和 JSON 字段名。")

    first = session.get(
        API_URL,
        params={PAGE_PARAM: 1, PAGE_SIZE_PARAM: 50},
        timeout=20,
        verify=False
    )
    first.raise_for_status()
    data = first.json()

    total = int(get_field(data, TOTAL_KEY))
    page_size = int(get_field(data, PAGE_SIZE_KEY))
    if page_size <= 0:
        raise ValueError(f"每页条数不合法：{page_size}")
    total_pages = (total + page_size - 1) // page_size

    all_items = []
    for page in range(1, total_pages + 1):
        resp = session.get(
            API_URL,
            params={PAGE_PARAM: page, PAGE_SIZE_PARAM: page_size},
            timeout=20,
            verify=False
        )
        resp.raise_for_status()
        page_data = resp.json()
        page_items = get_field(page_data, ITEMS_KEY)
        if not isinstance(page_items, list):
            raise TypeError(f"ITEMS_KEY={ITEMS_KEY!r} 对应的值不是 list，请检查 JSON 字段。")
        all_items.extend(page_items)
        print(f"第 {page}/{total_pages} 页：{len(page_items)} 条，累计 {len(all_items)} 条")

    return data, total, page_size, total_pages, all_items

# 取消下一行注释以开始采集
first_data, device_total, device_page_size, device_total_pages, devices = fetch_devices()

C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-pack

第 1/100 页：50 条，累计 50 条
第 2/100 页：50 条，累计 100 条
第 3/100 页：50 条，累计 150 条
第 4/100 页：50 条，累计 200 条
第 5/100 页：50 条，累计 250 条
第 6/100 页：50 条，累计 300 条


C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-pack

第 7/100 页：50 条，累计 350 条
第 8/100 页：50 条，累计 400 条
第 9/100 页：50 条，累计 450 条
第 10/100 页：50 条，累计 500 条
第 11/100 页：50 条，累计 550 条
第 12/100 页：50 条，累计 600 条


C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-pack

...

第 86/100 页：50 条，累计 4300 条
第 87/100 页：50 条，累计 4350 条
第 88/100 页：50 条，累计 4400 条
第 89/100 页：50 条，累计 4450 条
第 90/100 页：50 条，累计 4500 条


C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-pack

第 91/100 页：50 条，累计 4550 条
第 92/100 页：50 条，累计 4600 条
第 93/100 页：50 条，累计 4650 条
第 94/100 页：50 条，累计 4700 条
第 95/100 页：50 条，累计 4750 条


C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-pack

第 96/100 页：50 条，累计 4800 条
第 97/100 页：50 条，累计 4850 条
第 98/100 页：50 条，累计 4900 条
第 99/100 页：50 条，累计 4950 条
第 100/100 页：50 条，累计 5000 条


In [12]:

from pathlib import Path
import json

# 检查采集数量
print("设备总数：", len(devices))
print("API 返回总数：", device_total)

# 检查 device_id 是否唯一
device_ids = [item.get("device_id") for item in devices]
unique_ids = set(device_ids)

print("设备编号数量：", len(device_ids))
print("唯一设备编号数量：", len(unique_ids))
print("缺失设备编号数量：", sum(x is None for x in device_ids))
print("重复设备编号数量：", len(device_ids) - len(unique_ids))

# 创建 data 目录并保存 JSON
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

print("已保存：", (data_dir / "devices.json").resolve())


设备总数： 5000
API 返回总数： 5000
设备编号数量： 5000
唯一设备编号数量： 5000
缺失设备编号数量： 0
重复设备编号数量： 0
已保存： D:\大数据采集\实验一\lab01\data\devices.json


## 3. Company：静态 HTML 解析与字段提取

In [7]:
def fetch_company_page(url):
    response = session.get(url, timeout=20, verify=False)
    print("GET", url, "->", response.status_code)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser"), response.url

if COMPANY_CARD_SELECTOR:
    company_soup, current_company_url = fetch_company_page(COMPANY_URL)
    company_cards = company_soup.select(COMPANY_CARD_SELECTOR)
    print("第 1 页企业记录数：", len(company_cards))
    if company_cards:
        print("第一条记录 outerHTML（前 5000 字符）：")
        print(str(company_cards[0])[:5000])
else:
    print("请先在配置区填写 COMPANY_CARD_SELECTOR。")

请先在配置区填写 COMPANY_CARD_SELECTOR。


根据第一条记录的 `outerHTML` 确认实际字段位置，再运行/修改下面函数。函数会优先读取 `data-*` 属性中的 ID，并尝试从 `dt/dd` 中提取行业、区域、设备数量；企业名称、状态的 selector 需要根据实际 DOM 填写。

In [ ]:
# 按实际 DOM 填写；例如 .company-name 仅为写法示例，不能直接假定平台使用该 class
COMPANY_NAME_SELECTOR = ""
COMPANY_STATUS_SELECTOR = ""
COMPANY_ID_ATTRIBUTE = "data-company-id"  # 请按实际 outerHTML 修改

def parse_company_card(card):
    details = {}
    for item in card.select("dl > div"):
        dt = item.select_one("dt")
        dd = item.select_one("dd")
        if dt and dd:
            details[dt.get_text(" ", strip=True)] = dd.get_text(" ", strip=True)

    company_id = card.get(COMPANY_ID_ATTRIBUTE)
    name_node = card.select_one(COMPANY_NAME_SELECTOR) if COMPANY_NAME_SELECTOR else None
    status_node = card.select_one(COMPANY_STATUS_SELECTOR) if COMPANY_STATUS_SELECTOR else None

    # 下面的字段名映射需要按页面真实显示的 dt 文本调整
    industry = details.get("行业", details.get("所属行业", ""))
    zone = details.get("区域", details.get("所属区域", ""))
    device_count_raw = details.get("设备数量", details.get("设备数", ""))
    digits = "".join(ch for ch in str(device_count_raw) if ch.isdigit())
    device_count = int(digits) if digits else None

    return {
        "company_id": company_id,
        "name": name_node.get_text(" ", strip=True) if name_node else "",
        "industry": industry,
        "zone": zone,
        "device_count": device_count,
        "status": status_node.get_text(" ", strip=True) if status_node else "",
    }

if COMPANY_CARD_SELECTOR and COMPANY_NAME_SELECTOR:
    preview_soup, _ = fetch_company_page(COMPANY_URL)
    preview_cards = preview_soup.select(COMPANY_CARD_SELECTOR)
    if preview_cards:
        print(parse_company_card(preview_cards[0]))
    else:
        print("没有找到企业记录，请检查 COMPANY_CARD_SELECTOR。")
else:
    print("请填写 COMPANY_CARD_SELECTOR 和 COMPANY_NAME_SELECTOR，并按实际页面修改 ID / 状态 selector。")

## 4. 沿“下一页”链接采集 Company 前 3 页

In [22]:

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
import json

# 从企业列表第一页开始
url = COMPANY_URL
companies = []

for page in range(1, 4):
    response = company_session.get(
        url, timeout=20, verify=False
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    cards = soup.select("article.company-card")

    print(f"第 {page} 页：采集到 {len(cards)} 家企业")

    if not cards:
        raise ValueError(f"第 {page} 页没有找到企业卡片，请检查网页结构")

    for card in cards:
        company_id = card.get("data-company-id")
        name_tag = card.select_one("h2 a")
        name = name_tag.get_text(strip=True) if name_tag else None

        companies.append({
            "company_id": company_id,
            "name": name
        })

    # 前两页查找文字明确为“下一页”的链接
    if page < 3:
        nav = soup.select_one("nav.pagination")
        next_link = None

        if nav:
            next_link = next(
                (
                    a for a in nav.select("a")
                    if a.get_text(strip=True) == "下一页"
                ),
                None
            )

        if not next_link or not next_link.get("href"):
            raise ValueError(f"第 {page} 页没有找到“下一页”链接")

        url = urljoin(response.url, next_link["href"])
        print("下一页地址：", url)

# 检查企业编号
company_ids = [c["company_id"] for c in companies]

print("\n企业采集总数：", len(companies))
print("唯一企业编号数：", len(set(company_ids)))
print("缺失企业编号数：", sum(not x for x in company_ids))

# 保存结果
output_path = Path("data") / "companies.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("企业数据已保存到：", output_path.resolve())


C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,


第 1 页：采集到 20 家企业
下一页地址： https://115.159.159.82/companies?page=2
第 2 页：采集到 20 家企业
下一页地址： https://115.159.159.82/companies?page=3
第 3 页：采集到 20 家企业

企业采集总数： 60
唯一企业编号数： 60
缺失企业编号数： 0
企业数据已保存到： D:\大数据采集\实验一\lab01\data\companies.json


In [23]:

import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = COMPANY_URL
company_checks = []

for page_no in range(1, 4):
    response = company_session.get(
        url, timeout=20, verify=False
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    cards = soup.select(COMPANY_CARD_SELECTOR)

    for card in cards:
        company_id = card.get("data-company-id")
        text = card.get_text(" ", strip=True)

        match = re.search(
            r"设备数量\s*([0-9,]+)\s*台?",
            text
        )

        if match:
            registered_count = int(
                match.group(1).replace(",", "")
            )
            company_checks.append({
                "company_id": company_id,
                "registered_count": registered_count
            })
        else:
            print("未能提取设备数量：", company_id, text)

    if page_no < 3:
        nav = soup.select_one("nav.pagination")
        next_link = None

        if nav:
            next_link = next(
                (
                    a for a in nav.select("a")
                    if a.get_text(strip=True) == "下一页"
                ),
                None
            )

        if not next_link or not next_link.get("href"):
            raise ValueError(f"第 {page_no} 页找不到下一页")

        url = urljoin(response.url, next_link["href"])

# 使用前面已经采集好的 devices 列表
device_counts = {}

for device in devices:
    cid = device.get("company_id")
    if cid:
        device_counts[cid] = device_counts.get(cid, 0) + 1

mismatches = []

for company in company_checks:
    cid = company["company_id"]
    actual_count = device_counts.get(cid, 0)

    if actual_count != company["registered_count"]:
        mismatches.append({
            "company_id": cid,
            "网页登记数量": company["registered_count"],
            "实际设备数量": actual_count
        })

print("企业校验记录数：", len(company_checks))
print("唯一企业编号数：", len(set(
    c["company_id"] for c in company_checks
)))
print("成功提取设备数量的企业数：", len(company_checks))
print("数量不一致的企业数：", len(mismatches))
print("不一致详情：", mismatches[:10])


C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,
C:\Users\Pleasant Heart\lib\site-packages\urllib3\connectionpool.py:1068: InsecureRequestWarning: Unverified HTTPS request is being made to host '115.159.159.82'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  InsecureRequestWarning,


企业校验记录数： 60
唯一企业编号数： 60
成功提取设备数量的企业数： 60
数量不一致的企业数： 0
不一致详情： []


## 5. 验证数量、ID 唯一性与 Company–Device 一致性

In [ ]:
def validate_results(companies, devices):
    company_ids = [x.get("company_id") for x in companies]
    device_ids = [x.get("device_id") for x in devices]

    print("Company 总数：", len(companies))
    print("唯一 company_id 数：", len(set(x for x in company_ids if x not in (None, ""))))
    print("Device 总数：", len(devices))
    print("唯一 device_id 数：", len(set(x for x in device_ids if x not in (None, ""))))

    device_counts = Counter(x.get("company_id") for x in devices)
    mismatch = []
    for company in companies:
        cid = company.get("company_id")
        expected = company.get("device_count")
        actual = device_counts.get(cid, 0)
        if expected is not None and expected != actual:
            mismatch.append({"company_id": cid, "name": company.get("name"),
                             "expected": expected, "actual": actual})
    print("Company–Device 不一致企业数：", len(mismatch))
    if mismatch:
        print("不一致示例：", mismatch[:10])
    return mismatch

# 在 devices 和 companies 采集成功后运行：
# mismatches = validate_results(companies, devices)

## 6. 保存 JSON 文件

In [ ]:
def save_results(companies, devices):
    DATA_DIR.mkdir(exist_ok=True)
    with open(DATA_DIR / "companies.json", "w", encoding="utf-8") as f:
        json.dump(companies, f, ensure_ascii=False, indent=2)
    with open(DATA_DIR / "devices.json", "w", encoding="utf-8") as f:
        json.dump(devices, f, ensure_ascii=False, indent=2)
    print("已保存：", DATA_DIR / "companies.json")
    print("已保存：", DATA_DIR / "devices.json")

# 验证通过后运行：
# save_results(companies, devices)

## 7. 实验结果记录

手册中的目标值为：Company 60 条、60 个唯一 `company_id`；Device 5000 条、5000 个唯一 `device_id`；Company–Device 不一致企业数为 0。请以实际运行输出为准，不要为了匹配目标而手工修改结果。

请将报告要求的 4 张真实截图保存到 `images/`：
1. `01_device_api.png`：浏览器 Network / Fetch-XHR 的设备 API 请求及 Headers；
2. `02_company_source.png`：`/companies` 源代码中 Ctrl+F 搜索到真实企业名称；
3. `03_company_next.png`：Elements 中被选中的“下一页”链接及 href；
4. `04_validation.png`：JupyterLab 中最终验证数字。

**截图必须来自实际操作，不能用代码输出或伪造图片替代。**